# BBPE的从零开始实现
在 Python 中，这其实反而让代码变得更加简单和优雅，因为 Python 的 bytes 对象本质上就是一个取值范围在 0 到 255 之间的整数列表！我们不需要像之前那样处理繁琐的正则表达式和 </w> 结尾符。

核心实现思路：
1. 初始状态： 基础词表固定为 256 个（对应字节 00 到 FF，即十进制的 0 到 255）。
2. 训练阶段 (Train)：
   * 将输入的文本通过 .encode("utf-8") 转换为字节流（即一长串 0-255 的整数）。
   * 统计相邻数字对的频率。
   * 将频率最高的一对数字（例如 (100, 101)）替换为一个新的整数 ID（从 256 开始递增，比如 256）。
   * 记录这个合并规则，并更新词汇表。
   * 循环往复，直到达到我们设定的目标词表大小。
3. 推理阶段 (Encode / Decode)：
   * Encode: 把新文本转为基础字节列表，然后严格按照训练时记录的优先级，不断合并相邻的数字，直到无法合并为止。
   * Decode: 根据我们维护的词汇字典，把大 ID 重新展开成基础字节，最后用 .decode("utf-8") 还原成人类可读的文本。

In [1]:
# In[1]: 定义 BBPE 核心类

class BasicBBPE:
    def __init__(self):
        # merges 记录合并规则及其优先级: {(byte1, byte2): new_id}
        self.merges = {}
        # vocab 记录每个 ID 对应的实际字节序列: {id: bytes}
        # 初始词表包含 0-255 个基础字节
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        
    def _get_stats(self, ids):
        """内部方法：统计相邻元素对的出现频率"""
        counts = {}
        # zip(ids, ids[1:]) 是一个优雅的 Python 技巧，用于遍历所有相邻对
        for pair in zip(ids, ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def _merge(self, ids, pair, idx):
        """内部方法：在 ids 列表中，将出现的所有 pair 替换为新的 idx"""
        newids = []
        i = 0
        while i < len(ids):
            # 如果看当前元素和下一个元素正好匹配我们要找的 pair
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                newids.append(idx)
                i += 2 # 跳过被合并的两个元素
            else:
                newids.append(ids[i])
                i += 1
        return newids

    def train(self, text, vocab_size):
        """
        训练 BBPE 模型
        text: 训练语料 (字符串)
        vocab_size: 目标词表大小 (必须 >= 256)
        """
        assert vocab_size >= 256, "BBPE 的初始词表大小至少为 256"
        num_merges = vocab_size - 256
        
        # 1. 将文本编码为原始 UTF-8 字节列表 (0-255 的整数)
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)
        
        print(f"训练文本转化为底层字节后，共有 {len(ids)} 个 token。开始训练...")
        
        # 2. 开始迭代合并
        for i in range(num_merges):
            stats = self._get_stats(ids)
            if not stats:
                break # 如果没有可以合并的对了，提前结束
                
            # 找出频率最高的那一对
            best_pair = max(stats, key=stats.get)
            
            # 分配新的 token ID (从 256 开始递增)
            new_id = 256 + i
            
            # 在序列中进行实际的替换
            ids = self._merge(ids, best_pair, new_id)
            
            # 记录合并规则
            self.merges[best_pair] = new_id
            
            # 更新词汇表：新 token 的字节 = 左 token 字节 + 右 token 字节
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            
            if (i + 1) % 10 == 0 or i == num_merges - 1:
                print(f"Merge {i+1}/{num_merges}: {best_pair} -> {new_id} (出现 {stats[best_pair]} 次)")
                
        print(f"训练完成！当前词表大小: {len(self.vocab)}")

    def encode(self, text):
        """推理阶段：将输入文本转为 Token ID 列表"""
        # 1. 初始转换为基础字节
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)
        
        # 2. 循环合并，直到没有任何规则适用
        while len(ids) >= 2:
            stats = self._get_stats(ids)
            
            # 在当前序列存在的所有对中，找到在训练时“最早被合并”的那一对
            # self.merges 中记录的值就是它被合并的先后顺序 (越小越早)
            # 如果当前的对不在 merges 字典里，给它一个无穷大的值表示不合并
            pair = min(stats.keys(), key=lambda p: self.merges.get(p, float("inf")))
            
            # 如果找出的最优对都不在我们的规则库里，说明彻底合并不了了
            if pair not in self.merges:
                break
                
            # 执行合并
            new_id = self.merges[pair]
            ids = self._merge(ids, pair, new_id)
            
        return ids

    def decode(self, ids):
        """解码阶段：将 Token ID 列表还原为人类文本"""
        # 将所有的 ID 根据词汇表映射回原始的 bytes，然后拼接起来
        b = b"".join([self.vocab[idx] for idx in ids])
        # 将字节解码为字符串。errors="replace" 用于防止部分由于截断导致的非标准 UTF-8 报错
        text = b.decode("utf-8", errors="replace")
        return text

print("BBPE 核心类定义完毕！")

BBPE 核心类定义完毕！


In [2]:
# In[2]: 测试我们的 BBPE 引擎

# 1. 准备一段包含中英文和 Emoji 的极具挑战性的训练语料
corpus = """
自然语言处理 (NLP) 是一门充满魅力的学科。🔥
Language modeling is the core of LLMs. 🔥
自然语言处理需要处理海量的 token。
Tokenization is very important.
"""

# 初始化模型并设定目标词表大小为 280 (即合并 24 次)
bbpe = BasicBBPE()
bbpe.train(corpus, vocab_size=280)

# 2. 测试 Encode (推理预测)
test_text = "自然语言处理太棒了！🔥 NLP is core."
print("\n--- 测试编码 (Encode) ---")
print(f"原始文本: {test_text}")
encoded_ids = bbpe.encode(test_text)
print(f"编码后的 ID 序列: {encoded_ids}")

# 3. 测试 Decode (还原)
print("\n--- 测试解码 (Decode) ---")
decoded_text = bbpe.decode(encoded_ids)
print(f"解码后的文本: {decoded_text}")

# 4. 观察 Token 的具体形态
print("\n--- 查看编码的具体 Token 碎片 ---")
for idx in encoded_ids:
    raw_bytes = bbpe.vocab[idx]
    # 我们试着把每个 token 代表的字节打印出来看看
    try:
        readable = raw_bytes.decode("utf-8")
    except UnicodeDecodeError:
        readable = f"<不可打印的乱码碎片: {raw_bytes}>"
    print(f"ID {idx}: {readable}")

训练文本转化为底层字节后，共有 189 个 token。开始训练...
Merge 10/24: (264, 170) -> 265 (出现 2 次)
Merge 20/24: (274, 260) -> 275 (出现 2 次)
Merge 24/24: (278, 130) -> 279 (出现 2 次)
训练完成！当前词表大小: 280

--- 测试编码 (Encode) ---
原始文本: 自然语言处理太棒了！🔥 NLP is core.
编码后的 ID 序列: [232, 135, 170, 231, 132, 182, 232, 175, 173, 232, 168, 128, 260, 256, 170, 230, 163, 146, 228, 186, 134, 239, 188, 129, 240, 159, 148, 165, 32, 78, 76, 80, 262, 115, 32, 99, 111, 114, 101, 46]

--- 测试解码 (Decode) ---
解码后的文本: 自然语言处理太棒了！🔥 NLP is core.

--- 查看编码的具体 Token 碎片 ---
ID 232: <不可打印的乱码碎片: b'\xe8'>
ID 135: <不可打印的乱码碎片: b'\x87'>
ID 170: <不可打印的乱码碎片: b'\xaa'>
ID 231: <不可打印的乱码碎片: b'\xe7'>
ID 132: <不可打印的乱码碎片: b'\x84'>
ID 182: <不可打印的乱码碎片: b'\xb6'>
ID 232: <不可打印的乱码碎片: b'\xe8'>
ID 175: <不可打印的乱码碎片: b'\xaf'>
ID 173: <不可打印的乱码碎片: b'\xad'>
ID 232: <不可打印的乱码碎片: b'\xe8'>
ID 168: <不可打印的乱码碎片: b'\xa8'>
ID 128: <不可打印的乱码碎片: b'\x80'>
ID 260: 处理
ID 256: <不可打印的乱码碎片: b'\xe5\xa4'>
ID 170: <不可打印的乱码碎片: b'\xaa'>
ID 230: <不可打印的乱码碎片: b'\xe6'>
ID 163: <不可打印的乱码碎片: b'\xa3'>
ID 146